In [4]:
import torch
import pandas as pd

from CNN import InfernoCalibNet, ChestXRayDataset, OUT_DIR, CALIB_DIR


def predict_single_entry(index=1):
    torch.cuda.empty_cache()

    # Load metadata and single image entry
    df = pd.read_csv(OUT_DIR / "ml_test.csv")
    row = df.iloc[index]

    # Load model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = InfernoCalibNet(num_classes=2).to(device)
    model.load_state_dict(torch.load(CALIB_DIR / "InfernoCalibNetML.pth", weights_only=True))
    model.eval()

    # Manually create dataset with only one image
    ds = ChestXRayDataset(OUT_DIR / "ml_test.csv", transform=False)
    image_tensor, target_tensor = ds[index]
    image_tensor = image_tensor.unsqueeze(0).to(device)  # Add batch dim

    with torch.no_grad():
        output = model(image_tensor)
        logits = output.cpu().squeeze()
        probs = torch.sigmoid(logits)

    print("Index:", index)
    print("True Labels:    ", target_tensor.numpy())
    print("Logits:         ", logits.numpy())
    print("Sigmoid Confidence Scores:  ", probs.numpy())

    return logits.numpy(), probs.numpy(), target_tensor.numpy()

predict_single_entry()


Index: 1
True Labels:     [1. 0.]
Logits:          [ 0.11380506 -2.5969694 ]
Sigmoid Confidence Scores:   [0.52842057 0.06933372]


(array([ 0.11380506, -2.5969694 ], dtype=float32),
 array([0.52842057, 0.06933372], dtype=float32),
 array([1., 0.], dtype=float32))